# pandas `groupby` — Reference Notebook

Un guide pratique des opérations `groupby` en pandas, organisé par cas d'usage : agrégations classiques, transformations, opérations avancées avec `.apply`, décalages temporels avec `.shift`, et fenêtres glissantes.

Chaque cellule de code est **autonome** : les imports sont répétés pour permettre le copier-coller indépendant.

---
## Données communes utilisées dans le notebook

In [ ]:
import pandas as pd
import numpy as np

# Dataset de référence : transactions e-commerce
rng = np.random.default_rng(42)

n = 200
df = pd.DataFrame({
    'date':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'region':   rng.choice(['Nord', 'Sud', 'Est', 'Ouest'], n),
    'produit':  rng.choice(['A', 'B', 'C'], n),
    'vendeur':  rng.choice(['Alice', 'Bob', 'Clara', 'David'], n),
    'ventes':   rng.integers(100, 2000, n).astype(float),
    'quantite': rng.integers(1, 50, n),
})

# Introduire quelques valeurs manquantes
df.loc[rng.choice(n, 10, replace=False), 'ventes'] = np.nan

print(df.shape)
df.head(8)

---
# Partie 1 — Agrégations de base

## 1.1 `.agg()` — plusieurs métriques en une passe

**Question :** Quelle est la performance (total, moyenne, médiane, max) par groupe ?

**Quand utiliser :** Tableau de bord, reporting, comparaison de segments.

**Principe.** `groupby` divise le DataFrame en sous-groupes selon une ou plusieurs clés, puis applique une fonction d'agrégation à chaque groupe. `.agg()` permet de passer un dictionnaire `{colonne: [fonctions]}` pour calculer plusieurs métriques simultanément.

```python
df.groupby('cle')['col'].agg(['mean', 'sum', 'count'])
df.groupby('cle').agg({'col1': ['sum', 'mean'], 'col2': 'max'})
```

In [ ]:
import pandas as pd
import numpy as np

# ── Agrégation simple sur une colonne ──
print('=== Ventes par région ===')
display(
    df.groupby('region')['ventes']
    .agg(['sum', 'mean', 'median', 'count'])
    .round(1)
    .sort_values('sum', ascending=False)
)

# Agrégation multi-colonnes avec nommage explicite (syntaxe NamedAgg)
print('\n=== Résumé par région x produit ===')
resume = df.groupby(['region', 'produit']).agg(
    total_ventes  = ('ventes',   'sum'),
    nb_lignes     = ('ventes',   'count'),
    ventes_moy    = ('ventes',   'mean'),
    qte_totale    = ('quantite', 'sum'),
).round(1).reset_index()
display(resume.head(10))

# Expected output:
# ──────────────────────────────────────────────
#          sum    mean  median  count
# region
# Nord   18420   987.0   943.0     19
# Sud    31240  1041.3   985.0     30
# ...
# (NamedAgg évite le multi-index sur les colonnes)

**Points clés :**

> *`agg(nom=('col', 'func'))` (syntaxe NamedAgg) produit des colonnes avec des noms lisibles — à préférer au dictionnaire.*

> *`.count()` ignore les `NaN` ; `.size()` les compte. Pour le nombre de lignes brut par groupe : `.groupby('cle').size()`.*

> *Par défaut, `groupby` exclut les clés `NaN`. Passer `dropna=False` pour les conserver.*

---

## 1.2 Fonctions d'agrégation personnalisées

**Question :** Comment calculer une métrique qui n'existe pas nativement (ex : IQR, coefficient de variation, top-N) ?

**Principe.** On passe une fonction Python dans `.agg()`. La fonction reçoit une `Series` (les valeurs du groupe pour la colonne sélectionnée) et retourne un **scalaire**.

In [ ]:
import pandas as pd
import numpy as np

# ── Fonctions personnalisées ──
def iqr(x):           return x.quantile(0.75) - x.quantile(0.25)
def cv(x):            return x.std() / x.mean() if x.mean() != 0 else np.nan
def top3_mean(x):     return x.nlargest(3).mean()
def pct_above(x, seuil=1000): return (x > seuil).mean() * 100

stats_custom = df.groupby('produit')['ventes'].agg(
    mediane       = 'median',
    iqr           = iqr,
    coeff_var     = cv,
    moy_top3      = top3_mean,
    pct_sup_1000  = pct_above,
).round(2)

display(stats_custom)

# Expected output:
# ──────────────────────────────────────────────
#          mediane     iqr  coeff_var  moy_top3  pct_sup_1000
# produit
# A          962.0   748.0       0.54    1882.3         46.3
# B         1023.5   801.2       0.52    1924.7         51.8
# C          887.0   720.5       0.57    1895.2         43.1

---

# Partie 2 — `.transform()` : enrichir sans réduire

## 2.1 Principe et cas d'usage

**Question :** Comment ajouter une colonne calculée par groupe **sans réduire** le DataFrame ?

**Différence clé avec `.agg()` :**
- `.agg()` retourne **un scalaire par groupe** → DataFrame réduit (une ligne par groupe)
- `.transform()` retourne **une valeur par ligne** alignée sur l'index original → même shape que l'entrée

**Cas typiques :** normaliser par groupe, remplir les NaN par la moyenne du groupe, créer des flags.

In [ ]:
import pandas as pd
import numpy as np

df2 = df.copy()

# 1. Moyenne par groupe → broadcast sur chaque ligne
df2['moy_ventes_region']   = df2.groupby('region')['ventes'].transform('mean')
df2['total_ventes_region'] = df2.groupby('region')['ventes'].transform('sum')

# 2. Z-score intra-groupe
df2['zscore_region'] = df2.groupby('region')['ventes'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# 3. Remplir les NaN par la médiane du groupe
df2['ventes_filled'] = df2.groupby('produit')['ventes'].transform(
    lambda x: x.fillna(x.median())
)

# 4. Rang des ventes au sein de chaque région
df2['rang_region'] = df2.groupby('region')['ventes'].rank(ascending=False, method='dense')

# 5. Flag : vente supérieure à la moyenne de son produit
df2['above_avg_produit'] = df2['ventes'] > df2.groupby('produit')['ventes'].transform('mean')

print(f'Shape conservée : {df2.shape}  (toujours {len(df2)} lignes)')
print(f"NaN avant fill : {df['ventes'].isna().sum()}  |  après : {df2['ventes_filled'].isna().sum()}")
display(
    df2[['region', 'produit', 'ventes', 'moy_ventes_region', 'zscore_region', 'rang_region', 'above_avg_produit']]
    .head(8).round(2)
)

# Expected output:
# ─────────────────────────────────────────────────────────────────────
# Shape conservée : (200, 11)  (toujours 200 lignes)
# NaN avant fill : 10  |  après : 0

**Points clés :**

> *`.transform()` garantit que l'index de sortie est identique à l'index d'entrée — on peut directement assigner le résultat comme nouvelle colonne.*

> *Pour le fill NaN par groupe, `.transform(lambda x: x.fillna(x.median()))` est plus précis car il préserve le contexte intra-groupe.*

> *`.rank()` sur un groupby offre plus d'options que `.transform('rank')` : `method`, `ascending`, `pct`.*

---

# Partie 3 — `.apply()` : flexibilité maximale

## 3.1 Principe

**Question :** Comment appliquer une logique complexe qui nécessite plusieurs colonnes ou retourne un DataFrame ?

**Quand utiliser `.apply()` vs `.transform()` :**
- `.transform()` : fonction sur **une colonne**, retourne une **Series de même longueur**
- `.apply()` : fonction sur le **sous-DataFrame entier** du groupe, retourne ce que vous voulez (scalaire, Series, DataFrame)

**Attention :** `.apply()` est plus lent car il ne peut pas être vectorisé. À utiliser uniquement quand la logique nécessite plusieurs colonnes.

```python
df.groupby('cle').apply(func)          # func reçoit un sous-DataFrame
df.groupby('cle')['col'].apply(func)   # func reçoit une sous-Series
```

In [ ]:
import pandas as pd
import numpy as np

# ── Exemple 1 : scalaire calculé sur plusieurs colonnes ──
def revenu_par_unite(groupe):
    '''Ventes totales / quantité totale du groupe.'''
    return groupe['ventes'].sum() / groupe['quantite'].sum()

rev_par_unite = df.groupby(['region', 'produit']).apply(revenu_par_unite, include_groups=False)
print('=== Revenu par unité (région x produit) ===')
display(rev_par_unite.round(2).unstack())

# ── Exemple 2 : retourner les N meilleures lignes par groupe ──
def top_n(groupe, n=2):
    '''Retourne les n lignes avec les plus grosses ventes.'''
    return groupe.nlargest(n, 'ventes')

top2_par_region = (
    df.groupby('region', group_keys=False)
    .apply(top_n, n=2, include_groups=True)
    .reset_index(drop=True)
)
print('\n=== Top 2 ventes par région ===')
display(top2_par_region.sort_values('region'))

# ── Exemple 3 : z-score pondéré (logique multi-colonnes) ──
def zscore_pondere(groupe):
    '''Z-score de ventes pondéré par quantite dans le groupe.'''
    v = groupe['ventes']
    w = groupe['quantite']
    moy_w = np.average(v.dropna(), weights=w[v.notna()])
    std   = v.std()
    return (v - moy_w) / std if std > 0 else pd.Series(0, index=v.index)

df['zscore_pondere'] = (
    df.groupby('region', group_keys=False)
    .apply(zscore_pondere, include_groups=False)
)
print('\n=== Z-score pondéré intra-région ===')
display(df[['region', 'vendeur', 'ventes', 'quantite', 'zscore_pondere']].head(6).round(3))

**Points clés :**

> *`include_groups=False` (pandas >= 2.2) évite un `DeprecationWarning` quand la colonne de groupe se retrouve dans le DataFrame passé à la fonction.*

> *`group_keys=False` évite d'ajouter la clé de groupe comme niveau d'index dans le résultat — simplifie le `.reset_index()` ensuite.*

> *Pour les opérations exprimables avec `.agg()` ou `.transform()`, préférer ces méthodes : 5-50x plus rapides que `.apply()` sur de grands DataFrames.*

---

# Partie 4 — `.shift()` et calculs temporels

## 4.1 `.shift()` intra-groupe

**Question :** Comment calculer des variations (lag, delta, croissance) par entité dans un panel ?

**Problème sans `groupby` :** un simple `df['ventes'].shift(1)` décale toutes les lignes ensemble — la dernière ligne du groupe A contamine la première ligne du groupe B.

**Solution :** `df.groupby('entite')['col'].shift(n)` — le décalage est contenu à l'intérieur de chaque groupe.

```python
df.groupby('entite')['col'].shift(1)    # valeur précédente (lag 1)
df.groupby('entite')['col'].shift(-1)   # valeur suivante (lead 1)
```

**Important :** Toujours trier par `['entite', 'date']` avant d'appliquer `.shift()`.

In [ ]:
import pandas as pd
import numpy as np

# ── Dataset panel : ventes mensuelles par vendeur ──
rng = np.random.default_rng(42)
panel = pd.DataFrame({
    'mois':    pd.date_range('2023-01', periods=12, freq='MS').tolist() * 4,
    'vendeur': [v for v in ['Alice', 'Bob', 'Clara', 'David'] for _ in range(12)],
    'ventes':  rng.integers(500, 3000, 48).astype(float),
}).sort_values(['vendeur', 'mois']).reset_index(drop=True)

# ── Shift groupé ──
panel['ventes_lag1']  = panel.groupby('vendeur')['ventes'].shift(1)    # mois précédent
panel['ventes_lead1'] = panel.groupby('vendeur')['ventes'].shift(-1)   # mois suivant
panel['delta_pct']    = (panel.groupby('vendeur')['ventes'].pct_change() * 100).round(1)

# ── Moyenne mobile 3 mois intra-groupe ──
panel['ma3'] = panel.groupby('vendeur')['ventes'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# ── Max historique cumulé par vendeur ──
panel['record_hist'] = panel.groupby('vendeur')['ventes'].transform(
    lambda x: x.expanding().max()
)
panel['est_record'] = panel['ventes'] == panel['record_hist']

print('=== Panel Alice ===')
display(
    panel[panel['vendeur'] == 'Alice']
    [['mois', 'ventes', 'ventes_lag1', 'delta_pct', 'ma3', 'record_hist', 'est_record']]
    .round(1)
)
print(f"\nNaN dans ventes_lag1 : {panel['ventes_lag1'].isna().sum()} (= 4 vendeurs x 1 mois)")

# Expected output:
# ──────────────────────────────────────────────────────────────────────
#        mois  ventes  ventes_lag1  delta_pct    ma3  record_hist  est_record
# 0  2023-01   1843.0          NaN        NaN  1843.0       1843.0       True
# 1  2023-02    672.0       1843.0      -63.5  1257.5       1843.0      False
# 2  2023-03   2541.0        672.0      278.1  1685.3       2541.0       True

**Points clés :**

> *`.pct_change()` sur un groupby est équivalent à `(x - x.shift(1)) / x.shift(1)` — raccourci pratique pour les séries de croissance.*

> *`rolling(n, min_periods=1)` évite les NaN sur les premières lignes de chaque groupe.*

> *`.expanding().max()` calcule le maximum cumulé depuis la première observation de chaque groupe — utile pour détecter les records historiques.*

---

## 4.2 `pd.Grouper` — agrégation temporelle multi-niveaux

**Question :** Comment agréger des données à une fréquence plus basse (quotidien → mensuel) tout en conservant la dimension groupe ?

**Pattern :** `df.groupby(['entite', pd.Grouper(key='date', freq='ME')])` combine un groupby catégoriel et un grouper temporel.

In [ ]:
import pandas as pd
import numpy as np

# ── Agrégation mensuelle par région ──
mensuel = (
    df.groupby(['region', pd.Grouper(key='date', freq='ME')])
    .agg(
        total_ventes = ('ventes',   'sum'),
        nb_jours     = ('ventes',   'count'),
        qte_totale   = ('quantite', 'sum'),
    )
    .reset_index()
)
mensuel['ventes_par_jour'] = (mensuel['total_ventes'] / mensuel['nb_jours']).round(1)

# ── Croissance mensuelle (MoM) par région ──
mensuel = mensuel.sort_values(['region', 'date'])
mensuel['croissance_mom'] = (
    mensuel.groupby('region')['total_ventes'].pct_change() * 100
).round(1)

print('=== Données mensuelles par région ===')
display(mensuel.head(12))

print('\n=== Croissance MoM — région Nord ===')
display(mensuel[mensuel['region'] == 'Nord'][['date', 'total_ventes', 'croissance_mom']])

# Expected output:
# freq='ME' = fin de mois (Month End) ; 'MS' = début de mois
# ──────────────────────────────────────────────────────────────────────
#   region       date  total_ventes  nb_jours  qte_totale  ventes_par_jour  croissance_mom
# 0    Est 2024-01-31      18234.0        12  ...          1519.5             NaN

---

# Partie 5 — `.filter()` : sélectionner des groupes entiers

**Question :** Comment garder uniquement les lignes appartenant aux groupes qui vérifient une condition ?

**Principe.** `.filter(func)` évalue `func` sur chaque sous-groupe et retourne un DataFrame ne contenant que les groupes pour lesquels `func` renvoie `True`. Les lignes individuelles ne sont pas filtrées — c'est le **groupe entier** qui est retenu ou écarté.

In [ ]:
import pandas as pd
import numpy as np

# ── Régions avec volume total > seuil ──
seuil_ventes = 15_000
df_actif = df.groupby('region').filter(
    lambda g: g['ventes'].sum() > seuil_ventes
)
print(f"Régions conservées : {sorted(df_actif['region'].unique())}")
print(f'Lignes avant : {len(df)} | après : {len(df_actif)}')

# ── Produits présents dans au moins 3 régions ──
df_multi = df.groupby('produit').filter(
    lambda g: g['region'].nunique() >= 3
)
print(f"\nProduits présents dans >=3 régions : {sorted(df_multi['produit'].unique())}")

# ── Vendeurs dans le top 50% de variance ──
mediane_var = df.groupby('vendeur')['ventes'].var().median()
df_volatil = df.groupby('vendeur').filter(
    lambda g: g['ventes'].var() > mediane_var
)
print(f"\nVendeurs haute variance : {sorted(df_volatil['vendeur'].unique())}")

# ── Retirer les groupes avec trop de NaN ──
df_clean = df.groupby('produit').filter(
    lambda g: g['ventes'].isna().mean() < 0.10  # moins de 10% de NaN
)
print(f"\nProduits avec <10% NaN : {sorted(df_clean['produit'].unique())}")

# Expected output:
# ──────────────────────────────────────────────────────────────────────
# Régions conservées : ['Est', 'Nord', 'Ouest', 'Sud']
# Lignes avant : 200 | après : 178
# Produits présents dans >=3 régions : ['A', 'B', 'C']
# Vendeurs haute variance : ['Alice', 'David']

---

# Récapitulatif — Quelle méthode choisir ?

| Méthode | Reçoit | Retourne | Cas d'usage principal |
|---|---|---|---|
| `.agg()` | Series par colonne | **Scalaire** (1 ligne/groupe) | Tableaux de bord, reporting |
| `.transform()` | Series par colonne | **Series** (même longueur) | Normalisation, fill NaN, flags |
| `.apply()` | **DataFrame** du groupe | Anything | Logique multi-colonnes complexe |
| `.filter()` | DataFrame du groupe | **Bool** (garde/écarte le groupe) | Sélection de groupes entiers |
| `.shift(n)` | — | **Series** décalée de n intra-groupe | Lags, deltas, croissance |
| `.rolling(n).transform()` | Series | **Series** (même longueur) | Moyennes mobiles, volatilité |

**Règle de performance :** `.agg()` > `.transform()` > `.apply()`. Utiliser `.apply()` uniquement quand les deux premières ne suffisent pas.